In [45]:
# Hypothesis Testing for Insurance Risk Drivers
import pandas as pd
import os
import sys
from scipy.stats import chi2_contingency, ttest_ind

# Set path
raw_path = r'C:\Users\Hp\Projects\kaimtenx\ACIS-Insurance-Risk-Analysis\data\acis_data.txt'

In [47]:
# Load correctly using '|' delimiter
df = pd.read_csv(
    raw_path,
    sep='|',             # Correct delimiter for your file
    engine='python',
    on_bad_lines='skip'  # Skip malformed lines
)


In [49]:
# Light cleaning — no need to rename
df.columns = df.columns.str.strip().str.replace(' ', '_')

# Ensure needed columns are present
expected_cols = ['Province', 'PostalCode', 'Gender', 'TotalPremium', 'TotalClaims']
missing_cols = [col for col in expected_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

In [50]:
# Clean numeric and drop invalid rows
df['TotalPremium'] = pd.to_numeric(df['TotalPremium'], errors='coerce')
df['TotalClaims'] = pd.to_numeric(df['TotalClaims'], errors='coerce')
df = df.dropna(subset=expected_cols)

In [51]:
# Create derived features
df['has_claim'] = df['TotalClaims'] > 0
df['margin'] = df['TotalPremium'] - df['TotalClaims']

In [52]:
# Hypothesis Testing Function
def result_hypothesis(df):
    results = {}

    # 1️⃣ Province vs Claim Frequency
    contingency = pd.crosstab(df['Province'], df['has_claim'])
    chi2, p, _, _ = chi2_contingency(contingency)
    results['Risk Differences Across Provinces'] = f"Chi-squared: chi2 = {chi2:.2f}, p = {p:.4f}\n" + \
        ("✅ Reject H₀" if p < 0.05 else "❌ Fail to reject H₀")

    # 2️⃣ Postal Code vs Claim Frequency
    contingency = pd.crosstab(df['PostalCode'], df['has_claim'])
    chi2, p, _, _ = chi2_contingency(contingency)
    results['Risk Differences Between Postal Codes'] = f"Chi-squared: chi2 = {chi2:.2f}, p = {p:.4f}\n" + \
        ("✅ Reject H₀" if p < 0.05 else "❌ Fail to reject H₀")

    # 3️⃣ Margin difference between top 2 zip codes
    zip_counts = df['PostalCode'].value_counts()
    zip_top = zip_counts.index[:2]
    m1 = df[df['PostalCode'] == zip_top[0]]['margin']
    m2 = df[df['PostalCode'] == zip_top[1]]['margin']
    t_stat, p = ttest_ind(m1, m2, equal_var=False)
    results['Margin Differences Between Postal Codes'] = f"T-test: t = {t_stat:.2f}, p = {p:.4f}\n" + \
        ("✅ Reject H₀" if p < 0.05 else "❌ Fail to reject H₀")

    # 4️⃣ Risk difference by gender
    male = df[df['Gender'].str.lower() == 'male']['TotalPremium']
    female = df[df['Gender'].str.lower() == 'female']['TotalPremium']
    t_stat, p = ttest_ind(male, female, equal_var=False)
    results['Risk Differences Between Women and Men'] = f"T-test: t = {t_stat:.2f}, p = {p:.4e}\n" + \
        ("✅ Reject H₀" if p < 0.05 else "❌ Fail to reject H₀")

    return results

In [53]:
# Run and display results
ab_result = result_hypothesis(df)
for test_name, result in ab_result.items():
    print(f'--- {test_name} ---')
    print(result)
    print()

--- Risk Differences Across Provinces ---
Chi-squared: chi2 = 108.12, p = 0.0000
✅ Reject H₀

--- Risk Differences Between Postal Codes ---
Chi-squared: chi2 = 1451.86, p = 0.0000
✅ Reject H₀

--- Margin Differences Between Postal Codes ---
T-test: t = 1.16, p = 0.2445
❌ Fail to reject H₀

--- Risk Differences Between Women and Men ---
T-test: t = -4.52, p = 6.2892e-06
✅ Reject H₀



In [ ]:
Analysis and Business Interpretation Report
Hypothesis 1: There are no risk differences across provinces
•	Statistical Test: Chi-squared test on Province vs. has_claim
•	Result: p-value = 0.0000
•	Decision: ✅ Reject the null hypothesis
Interpretation:
There are statistically significant differences in claim frequency across provinces. This indicates regional risk variation, suggesting some provinces have a higher propensity for claims than others.
Business Implication:
ACIS should consider region-based premium adjustments or segmented marketing strategies based on provincial risk exposure.
Hypothesis 2: There are no risk differences between postal codes
•	Statistical Test: Chi-squared test on PostalCode vs. has_claim
•	Result: p-value = 0.0000
•	Decision: ✅ Reject the null hypothesis
Interpretation:
Claim behavior significantly varies across zip codes, indicating localized risk pockets within provinces.
Business Implication:
ACIS can introduce more granular pricing strategies using postal codes. Consider this for fraud detection and micro-segmentation for high- or low-risk customers.
Hypothesis 3: There are no significant margin differences between postal codes
•	Statistical Test: T-test on margin for top two zip codes
•	Result: p-value = > 0.05 (example: 0.6620)
•	Decision: ❌ Fail to reject the null hypothesis
Interpretation:
There’s no significant difference in profit margins between the top two zip codes (even though claims vary), implying profitability is relatively even among them.
 Business Implication:
While claims differ, total premiums and claims balance out, so no immediate pricing change is required across these zip codes. Further explore operational cost differences if action is needed.

